In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

from torch.utils.data import DataLoader

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cpu


In [3]:
import os

print(os.path.exists("./data/cifar-10-batches-py"))

False


In [4]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5)
    )
])

train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))

100%|██████████| 170M/170M [31:01<00:00, 91.6kB/s]


Training samples: 50000
Testing samples: 10000


In [5]:
#Experiment 1 CNN Model
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x


model = CNN()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model)
print("\nLearning Rate: 0.001")
print("Batch Size: 64")
print("Epochs: 3")

CNN(
  (conv_layers): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc_layers): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)

Learning Rate: 0.001
Batch Size: 64
Epochs: 3


In [6]:
#Experiment 1 Training
epochs = 3

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    average_loss = running_loss / len(train_loader)

    print(f"Epoch [{epoch + 1}/{epochs}], Loss: {average_loss:.4f}")

Epoch [1/3], Loss: 1.3321
Epoch [2/3], Loss: 0.9575
Epoch [3/3], Loss: 0.7991


In [7]:
#test accuracy
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy_exp1 = 100 * correct / total

print(f"Experiment 1 Test Accuracy: {accuracy_exp1:.2f}%")

Experiment 1 Test Accuracy: 69.40%


In [8]:
#Experiment 2 — Learning Rate Change.
model = CNN()

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.01
)

print("Experiment 2")
print("Learning Rate: 0.01")
print("Batch Size: 64")
print("Epochs: 3")

Experiment 2
Learning Rate: 0.01
Batch Size: 64
Epochs: 3


In [9]:
#training
epochs = 3

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    average_loss = running_loss / len(train_loader)

    print(f"Epoch [{epoch + 1}/{epochs}], Loss: {average_loss:.4f}")

Epoch [1/3], Loss: 2.3071
Epoch [2/3], Loss: 2.3035
Epoch [3/3], Loss: 2.3036


In [10]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy_exp2 = 100 * correct / total

print(f"Experiment 2 Test Accuracy: {accuracy_exp2:.2f}%")

Experiment 2 Test Accuracy: 10.00%


In [11]:
#Experiment 3 — Batch Size Change.
model = CNN()

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

train_loader_exp3 = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

test_loader_exp3 = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False
)

print("Experiment 3")
print("Learning Rate: 0.001")
print("Batch Size: 128")
print("Epochs: 3")

Experiment 3
Learning Rate: 0.001
Batch Size: 128
Epochs: 3


In [12]:
#training exp 3
epochs = 3

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader_exp3:
        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    average_loss = running_loss / len(train_loader_exp3)

    print(f"Epoch [{epoch + 1}/{epochs}], Loss: {average_loss:.4f}")

Epoch [1/3], Loss: 1.4094
Epoch [2/3], Loss: 1.0336
Epoch [3/3], Loss: 0.8796


In [13]:
#test accuracy
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader_exp3:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy_exp3 = 100 * correct / total

print(f"Experiment 3 Test Accuracy: {accuracy_exp3:.2f}%")

Experiment 3 Test Accuracy: 67.55%


## Hyperparameter Experiment Results

Three experiments were conducted using the same CNN architecture on the CIFAR-10 dataset.

| Experiment | Learning Rate | Batch Size | Epochs | Final Loss | Test Accuracy |
|------------|---------------|------------|--------|------------|---------------|
| Baseline | 0.001 | 64 | 3 | 0.7991 | 69.40% |
| Higher Learning Rate | 0.01 | 64 | 3 | 2.3036 | 10.00% |
| Larger Batch Size | 0.001 | 128 | 3 | 0.8796 | 67.55% |

## Conclusion

The baseline configuration with a learning rate of 0.001 and batch size of 64 achieved the best test accuracy of 69.40% among the three experiments.

Increasing the learning rate to 0.01 caused the model performance to drop significantly to 10.00%, indicating that the learning rate was too high for this configuration.

Increasing the batch size from 64 to 128 resulted in a slightly lower accuracy of 67.55%.

Therefore, among the tested configurations, a learning rate of 0.001 and batch size of 64 provided the best performance.